# Huấn luyện trên Google Colab _ Leanbot Detection

## Bước 1: Cài đặt thư viện và phụ thuộc

In [ ]:
!pip install -q ultralytics
import ultralytics
from IPython.display import Image, display
import os
ultralytics.checks()

Tải file dataset.zip


## Bước 2: Giải nén Dataset
- Sau khi tạo dataset bằng tool autolabel thì chỉ cần zip folder dataset trong đó chứa images và labels , sau đó upload lên server của Colab

In [ ]:
zip_path = '/content/datasets.zip'
if not os.path.exists(zip_path):
    print("Không tìm thấy datasets.zip!")
else:
    !unzip -q {zip_path} -d /content/
    print("Giải nén thành công.")

!ls -d /content/images

In [ ]:
import os
# Kiểm tra xem sau khi giải nén có những thư mục nào
print("Danh sách các tệp và thư mục trong /content:")
!ls -F /content/

#### Phân chia lại dataset : 70% train - 20% validate - 10% test

In [ ]:
import os
import random
import shutil
from collections import defaultdict

# Đường dẫn gốc
dataset_path = '/content/datasets'
images_path = os.path.join(dataset_path, 'images')
labels_path = os.path.join(dataset_path, 'labels')

# Tạo cấu trúc thư mục mới
for split in ['train', 'val', 'test']:\
    os.makedirs(os.path.join(dataset_path, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dataset_path, split, 'labels'), exist_ok=True)

# 1. Phân loại ảnh theo Class dựa trên file nhãn
class_groups = defaultdict(list)
all_images = [f for f in os.listdir(images_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

for img_f in all_images:
    label_f = img_f.rsplit('.', 1)[0] + '.txt'
    label_p = os.path.join(labels_path, label_f)
    
    if os.path.exists(label_p):
        with open(label_p, 'r') as f:
            lines = f.readlines()
            if lines:
                # Lấy class ID của đối tượng đầu tiên trong ảnh
                cls_id = lines[0].split()[0]
                class_groups[cls_id].append(img_f)
            else:
                class_groups['empty'].append(img_f)
    else:
        class_groups['no_label'].append(img_f)

# 2. Chia riêng từng Class và gộp lại
train_files, val_files, test_files = [], [], []

for cls_id, files in class_groups.items():
    random.shuffle(files)
    n = len(files)
    idx1 = int(n * 0.7)
    idx2 = idx1 + int(n * 0.2)
    
    train_files.extend(files[:idx1])
    val_files.extend(files[idx1:idx2])
    test_files.extend(files[idx2:])
    print(f"Class {cls_id}: Total={n}, Train={idx1}, Val={idx2-idx1}, Test={n-idx2}")

# 3. Di chuyển file
def move_files(files, split):
    for f in files:
        shutil.move(os.path.join(images_path, f), os.path.join(dataset_path, split, 'images', f))
        label_f = f.rsplit('.', 1)[0] + '.txt'
        if os.path.exists(os.path.join(labels_path, label_f)):
            shutil.move(os.path.join(labels_path, label_f), os.path.join(dataset_path, split, 'labels', label_f))

move_files(train_files, 'train')
move_files(val_files, 'val')
move_files(test_files, 'test')

print(f"\n✅ Đã chia xong: {len(train_files)} train, {len(val_files)} val, {len(test_files)} test.")


## Bước 3: Tạo File Cấu Hình và Script Huấn Luyện

In [ ]:
# 1. Tạo file train.py
train_py_content = """
import argparse
from ultralytics import YOLO

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', default='leanbot_data.yaml')
    parser.add_argument('--epochs', type=int, default=50)
    parser.add_argument('--batch', type=int, default=16)
    parser.add_argument('--name', default='leanbot_colab')
    return parser.parse_args()

if __name__ == '__main__':
    args = parse_args()
    model = YOLO('yolov8n.pt')
    model.train(
        data=args.data,
        epochs=args.epochs,
        batch=args.batch,
        device=0,
        name=args.name,
        degrees=10.0, fliplr=0.5, flipud=0.1
    )
"""
with open('train.py', 'w') as f:
    f.write(train_py_content.strip())

# 2. Tạo file leanbot_data.yaml
# Cập nhật đường dẫn đến các thư mục train/val cụ thể
yaml_content = """
path: /content/datasets
train: train/images
val: val/images
test: test/images
nc: 2
names:
  0: Leanbot_front
  1: Leanbot_back
"""
with open('leanbot_data.yaml', 'w') as f:
    f.write(yaml_content.strip())
print(" Đã cấu hình lại file YAML với đường dẫn train/val mới.")

## Bước 4: Bắt đầu Huấn Luyện

In [ ]:
!python train.py --epochs 100 --batch 16

## Bước 5: Đánh giá mô hình bằng Hình ảnh và Biểu đồ


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display
import os

# Cập nhật đường dẫn mới nhất từ lần chạy thứ 6
result_dir = Path('/content/runs/detect/leanbot_colab')

def show_image(filenames, title):
    if isinstance(filenames, str): filenames = [filenames]
    found = False
    for fname in filenames:
        img_path = result_dir / fname
        if os.path.exists(img_path):
            print(f"\n--- {title} ({fname}) ---")
            display(Image(filename=str(img_path), width=800))
            found = True
            break
    if not found:
        print(f"⚠️ Không tìm thấy biểu đồ cho: {title} (Thử tìm: {filenames})")

show_image('results.png', "Biểu đồ Kết quả Huấn luyện")
show_image(['confusion_matrix_normalized.png', 'confusion_matrix.png'], "Ma trận nhầm lẫn")
show_image('val_batch0_labels.jpg', "Ảnh gốc có Label")
show_image('val_batch0_pred.jpg', "Ảnh mô hình tự dự đoán")
show_image(['F1_curve.png', 'BoxF1_curve.png'], "Đường cong F1-Score")
show_image(['PR_curve.png', 'BoxPR_curve.png'], "Đường cong Precision-Recall")

## Bước 6: Chạy thử Inference (Dự đoán) trên ảnh mẫu


In [ ]:
from ultralytics import YOLO
import glob
import cv2
from google.colab.patches import cv2_imshow
import os


model_path = '/content/runs/detect/leanbot_colab/weights/best.pt'

if os.path.exists(model_path):
    model = YOLO(model_path)
    # Lấy ảnh từ tập test để chạy thử dự đoán
    sample_images = glob.glob('/content/datasets/test/images/*.jpg')

    if sample_images:
        print(f" Đang chạy dự đoán với model: {model_path}")
        # Giữ conf=0.25 vì mAP của bạn rất cao, không cần giảm quá thấp
        results = model.predict(source=sample_images, conf=0.25)
        for result in results:
            res_plotted = result.plot()
            cv2_imshow(res_plotted)
            print(f"Kết quả cho: {os.path.basename(result.path)}")
    else:
        print(" Không tìm thấy ảnh trong thư mục /content/datasets/test/images/")
else:
    print(f" Không tìm thấy model tại: {model_path}")

## Bước 7: Tải về Model Tốt Nhất

In [ ]:
from google.colab import files
import os

# Cập nhật đường dẫn tải về model từ lần chạy 6
best_model = '/content/runs/detect/leanbot_colab/weights/best.pt'
if os.path.exists(best_model):
    files.download(best_model)
    print("Đang chuẩn bị tải file model tốt nhất về máy...")
else:
    print(f" Không tìm thấy {best_model} để tải")